## Importing libraries

In [1]:
import magpylib as magpy
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.spatial import ConvexHull


## Initial parameters

In [10]:
resolution=1000    #number of points
R=40e-3            #in m
layers=1
diameter=1e-3      #diameter of individual wire
turns=20           #number of turns
plotting=0
current=2          #in A

fig = go.Figure()

## Defining spherical coil

In [3]:
def create_spherical_coil(R,turns, layers ,diameter,current,plotting):
    
    coils=[]
    coordinates=[]
    layer_current=current/layers
    
    for i in range (layers):
        
        t=np.linspace(0,1,resolution)
        phi=np.arccos(1-2*t)          #uniform z winding
        theta =2*np.pi*turns*t
        
        lx=(R+i*diameter)*np.cos(theta)*np.sin(phi)
        ly=(R+i*diameter)*np.sin(theta)*np.sin(phi)
        lz=(R+i*diameter)*np.cos(phi)

        layer_coordinates=np.stack((lx,ly,lz),axis=1)
        
        layer_coil=magpy.current.Polyline(current=layer_current,vertices=layer_coordinates)
        coils.append(layer_coil)
        coordinates.append(layer_coordinates)
        
        if plotting==True:
            fig.add_trace(go.Scatter3d(
            x=layer_coordinates[:, 0], y=layer_coordinates[:, 1], z=layer_coordinates[:, 2],
            mode='lines', name=f'Layer {i+1}'))

        
        
    if plotting==True:        
        fig.show(renderer="browser")
    
    
    top_lead = magpy.current.Polyline(current=current, vertices=np.array([[0, 0, 1.75 * R], [0, 0, R]]))
    bottom_lead = magpy.current.Polyline(current=current, vertices=np.array([[0, 0, -R], [0, 0, -1.75 * R]]))
    
    coils.extend([top_lead, bottom_lead])
    
    
    coordinates = np.vstack(coordinates)   #to convert list to numpy array
    
    
    return magpy.Collection(coils), coordinates

## Cost function

In [4]:
def cost_function(mags,roi): #ratio of the radius)

    roi_grid=np.mgrid[-roi*R:roi*R:20j, -roi*R:roi*R:20j, -roi*R:roi*R:20j].T
    

    r = np.linalg.norm(roi_grid, axis=-1)
    mask = r <= roi*R

    sphere_points = roi_grid[mask]
    
    B_roi=mags.getB(sphere_points)
    
    magnitude_B=np.linalg.norm(B_roi,axis=-1)
    
    mean_B=np.mean(magnitude_B)
    
    B_max=magnitude_B.max()
    B_min=magnitude_B.min()
    eta=((B_max-B_min)/mean_B)*1e6
    
    return eta, mean_B*1000, sphere_points


## Visualize in 3D

In [5]:
def visualize_3d(grid,mags):    #grid to be (N,N,N,3)
    sens = magpy.Sensor(pixel=grid)

    # hide axes arrows
    sens.style.arrows.x.show = False
    sens.style.arrows.y.show = False
    sens.style.arrows.z.show = False
    
    # control size
    sens.style.size = 2
    
    # pixel field settings
    sens.style.pixel.field.source = "B"
    sens.style.pixel.field.sizescaling = "linear"
    sens.style.pixel.field.colormap = "Inferno"
    sens.style.pixel.field.colorscaling = "linear"
    sens.style.pixel.field.symbol = "arrow3d"
    
    
    fig = magpy.show([sens, mags], backend="plotly", return_fig=True)
    fig.show(renderer="browser")

## Creating a coil

In [11]:
coils,coordinates=create_spherical_coil(R, turns, layers, diameter, current,plotting)  #coils is a mags object
roi=0.125
grid_3D=np.mgrid[-R*2:R*2:10j, -R*2:R*2:10j, -R*2:R*2:10j].T
eta,mean_B,sphere_points=cost_function(coils,roi)

## Results

In [7]:
visualize_3d(grid_3D,coils)

print('For Spherical','\n')
print(f'homogeneity ',eta,'\n')
print(f'field (mT) ',mean_B,'\n')


For Spherical 

homogeneity  87.36298198652013 

field (mT)  0.4090351402641633 



In [8]:
coils.show(renderer='browser')

## Exporting `.csv` file

In [12]:
np.savetxt("coordinates.csv", coordinates, delimiter=",")